#### Dependencies

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

from lifelines import KaplanMeierFitter, CoxPHFitter, NelsonAalenFitter
from sklearn.model_selection import KFold
from scipy.stats import gaussian_kde, norm
from sklearn.model_selection import train_test_split
from matplotlib.ticker import FuncFormatter
from lifelines.plotting import add_at_risk_counts
from lifelines.utils import concordance_index

from __future__ import annotations
from collections import Counter
from dataclasses import dataclass
import hashlib
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from lifelines.statistics import logrank_test
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None) 

#### Datasets

In [2]:
lical0 = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFile/lical0_processed_data_for_fp_model_21-01-2026.csv')
miro0 = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFile/miro0_processed_data_for_fp_model_21-01-2026.csv')
# ril_3010 = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFile/ril_3010_processed_data_for_fp_model_21-01-2026.csv')

# miroli0 = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFile/miroli0_processed_data_for_fp_model_21-01-2026.csv')
proact0 = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFile/proact0_processed_data_for_fp_model_21-01-2026.csv')

MND_lica = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFIle/MNDRegisterDataset_licals.csv')
MND_miro = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFIle/MNDRegisterDataset_mirocals.csv')
# MND_rilu = pd.read_csv('/Users/Apple/projects/ALS_Digital_Twins/All_processed_data/DataFIle/MNDRegisterDataset_riluzole.csv')

In [3]:
# lical0 = lical0.drop(columns=['slope', 'Age_VC', 'Sex_VC', 'Onset_VC', 'Vital_capacity'])
# miro0 = miro0.drop(columns=['slope', 'Age_VC', 'Sex_VC', 'Onset_VC', 'Vital_capacity'])


# MND_lica = MND_lica.drop(columns=['TRICALS_Risk_Score'])
# MND_miro = MND_miro.drop(columns=['TRICALS_Risk_Score'])

# lical0 = lical0.rename(columns={'Study_Duration': 'Disease_Duration'})
# miro0 = miro0.rename(columns={'Study_Duration': 'Disease_Duration'})
# proact0 = proact0.rename(columns={'Study_Duration': 'Disease_Duration'})
# MND_miro = MND_miro.rename(columns={'Disease_DurationD': 'Study_Duration', 'Enrollment_delay':'Enrollment_Delay'})

In [4]:
lical0.head(2)

,subject_id,Event,Vital_capacity,Age_Onset,Diagnostic_Delay,Study_Duration,Age_Diag,Enrollment_Delay,ALSFRS_RT,ALSFRS_Rasch,slope,Disease_DurationD,Onset_site_Limb,Sex_Male,Study_Arm_Placebo
0,P01001,0,107.0,51.967146,5.880420,19.000000,52.457221,27.660972,35.0,25.7,-0.684211,30.958607,1,1,1
1,P01002,1,99.0,59.613963,9.526938,19.528252,60.407940,9.034166,43.0,32.2,-0.256039,27.562418,1,0,1


In [5]:
MND_lica.head(2)

,subject_id,Event,Disease_DurationD,Age_Onset,Age_Diag,Diagnostic_Delay,Sex_Male,Onset_Limb,slope,Enrollment_Delay
0,87,1,27.726675,67.43,68.290000,10.381078,1,1,-0.530175,143.00
1,1086,1,17.575558,64.43,64.739998,3.646518,0,0,-0.836389,252.27


In [6]:
proact0.head(2)

,subject_id,Event,Vital_capacity,Age_Onset,Age_Diag,Diagnostic_Delay,Study_Duration,slope,ALSFRS_Rasch,ALSFRS_RT,Enrollment_Delay,Disease_DurationD,Onset_site_Limb,Sex_Male,Study_Arm_Placebo
0,121,1,63.037137,50.997947,51.584887,7.042707,504.0,-1.196093,21.1,27.0,4.980946,22.538108,1,0,0
1,226,1,56.228941,70.743326,71.594689,10.215506,393.0,-1.581523,20.6,26.0,4.863338,18.773982,0,1,0


In [7]:
# proact0 = proact0[proact0['Disease_Duration'] < 120]
lical0 = lical0.rename(columns={'Onset_site_Limb': 'Onset_Limb'})
miro0 = miro0.rename(columns={'Onset_site_Limb': 'Onset_Limb'})
proact0 = proact0.rename(columns={'Onset_site_Limb': 'Onset_Limb'})

In [ ]:


df = df_full.copy()

# Define your grouping variable (e.g., survival outcome, treatment)
group_var = "Event"   # <-- change as needed

# -------------------------------------------------------
# Detect variable types
# -------------------------------------------------------
continuous_vars = df[['Event', 'Age', 'Height', 'Weight', 'Study_Duration_months', 'ALSFRS_Rasch',
       'Diagnosis_delay', 'Vital_Capacity', 'TRICALS']].columns.drop(group_var, errors='ignore')

categorical_vars = df[['Sex', 'Onset_site', 'Riluzole_use', 'Study_Arm', 'European',  'trial_group']].columns

print(continuous_vars.tolist())
print('#-------------------------------------------------------#')
print(categorical_vars.tolist())

In [ ]:
# Optionally treat small-unique-number numeric vars as categorical
for col in continuous_vars:
    if df[col].nunique() <= 10:
        categorical_vars = categorical_vars.append(pd.Index([col]))
continuous_vars = [c for c in continuous_vars if c not in categorical_vars]

# -------------------------------------------------------
# Helper summary functions
# -------------------------------------------------------
def summarize_continuous(series):
    """Return Median [IQR]"""
    median = series.median()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    return f"{median:.2f} [{q1:.2f}, {q3:.2f}]"

def summarize_categorical(series):
    """Return n (%)"""
    counts = series.value_counts(dropna=False)
    total = len(series)
    return "; ".join([f"{cat}: {count} ({count/total*100:.1f}%)" for cat, count in counts.items()])

# -------------------------------------------------------
# Statistical tests (p-values)
# -------------------------------------------------------
def get_p_value(df, var, group_var):
    groups = [x.dropna() for _, x in df.groupby(group_var)[var]]
    if len(groups) != 2:
        return np.nan  # only supports 2 groups for now
    g1, g2 = groups

    # Continuous variable -> Mann-Whitney U
    if np.issubdtype(df[var].dtype, np.number):
        try:
            p = stats.mannwhitneyu(g1, g2, alternative='two-sided').pvalue
        except Exception:
            p = np.nan
    else:
        # Categorical variable -> Chi-square or Fisher's exact
        table = pd.crosstab(df[var], df[group_var])
        if table.shape == (2, 2):
            _, p = stats.fisher_exact(table)
        else:
            _, p, _, _ = stats.chi2_contingency(table, correction=False)
    return p

# -------------------------------------------------------
# Build the summary table
# -------------------------------------------------------
summary = {}

# Overall
overall = {}
for col in continuous_vars:
    overall[col] = summarize_continuous(df[col])
for col in categorical_vars:
    overall[col] = summarize_categorical(df[col])
summary["Overall"] = pd.Series(overall)

# Group-specific summaries
for group in sorted(df[group_var].dropna().unique()):
    subset = df[df[group_var] == group]
    group_summary = {}
    for col in continuous_vars:
        group_summary[col] = summarize_continuous(subset[col])
    for col in categorical_vars:
        group_summary[col] = summarize_categorical(subset[col])
    summary[f"{group_var}={group}"] = pd.Series(group_summary)


In [ ]:

# p-values
pvals = {}
for col in list(continuous_vars) + list(categorical_vars):
    pvals[col] = get_p_value(df, col, group_var)

summary["p-value"] = pd.Series(pvals)


# Combine into DataFrame
summary_table = pd.DataFrame(summary)
summary_table.index.name = "Variable"

# Format p-values
summary_table["p-value"] = summary_table["p-value"].apply(
    lambda x: f"{x:.3f}" if pd.notnull(x) else ""
)

# -------------------------------------------------------
# Display and save
# -------------------------------------------------------
summary_table
# summary_table.to_excel("/Users/Apple/projects/ALS_Digital_Twins/01_DigitalTwin_2/Results_20_10_2025/11.1.2_table1_survival_analysis.xlsx", index=True)
